In [1]:
# installing openai
!pip install "openai>=1.50.0" PyMuPDF requests matplotlib numpy pandas seaborn -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 23.0 MB/s eta 0:00:00
Dependencies installed.


In [2]:
from google.colab import userdata
Openai_API_KEY = userdata.get('Openai_API_KEY')
newsData_API_KEY = userdata.get('newsData_API_KEY')
Semantic_API_KEY = userdata.get('Semantic_API_KEY')



In [3]:
%matplotlib inline
import os, json, time, re, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
from openai import OpenAI
from IPython.display import Image, display

client = OpenAI(api_key=Openai_API_KEY)
print(f'   OpenAI version: {__import__("openai").__version__}')

   OpenAI version: 2.32.0


In [4]:
last_fetched_article = {"text": "", "title": "", "source": "", "date": ""}

In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
DRIVE_FOLDER = ["/content/drive/MyDrive/Colab Notebooks/Ancient Wisdom Corpus",
"/content/drive/MyDrive/Colab Notebooks/Bridging corpus"]

In [8]:
# I create nicknames for the path to each folder
ANCIENT_PATH = DRIVE_FOLDER[0]
BRIDGE_PATH = DRIVE_FOLDER[1]

In [7]:

              # 1. I first divide the files from each path so that Python can find its way to each folder.
ancient_files = [
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book I_Defining eudaimonia and the aim of Ethics.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book II-III_Excellence of character or moral virtue -voluntary and involuntary actions.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book IV_second set of examples of moral virtues.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book V_Justice and Fairness.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book VI_Intellectual Virtues.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book VII_Impediments to virtue.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Books VIII and IX_Friendship.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_NichomacheanEthics-Book X_Pleasure, happiness, and upbringing.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric_1.Foundations and History.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric_2.Mutual Relations and Core Concepts.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric_3.Analysis of Rhetoric Book I & Appendices.pdf"),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric_4.Analysis of Rhetoric Book II.pdf" ),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric_5.Analysis of Rhetoric Book III.pdf" ),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric_6.Appendices to Book III.pdf" ),
    os.path.join(ANCIENT_PATH, "Aristotle_Rhetoric.7_Rhetorica ad Alexandrum.pdf" ),
    os.path.join(ANCIENT_PATH, "Buddharakkhita_Dhammapada-1.Introduction and Basics.pdf" ),
    os.path.join(ANCIENT_PATH, "Buddharakkhita_Dhammapada-2.The Path of the Individual.pdf" ),
    os.path.join(ANCIENT_PATH, "Buddharakkhita_Dhammapada-3.Higher Teachings and Virtues.pdf" ),
    os.path.join(ANCIENT_PATH, "Buddharakkhita_Dhammapada-4.Final considerations.pdf" ),
    os.path.join(ANCIENT_PATH, "Epitectus_TheEnchiridion.pdf" ),
    os.path.join(ANCIENT_PATH, "LaoTsu_TaoTeChing.pdf" ),
    os.path.join(ANCIENT_PATH, "Lucretius_On the Nature of things.1_Book I Matter and the Void.pdf" ),
    os.path.join(ANCIENT_PATH, "Lucretius_On the Nature of things.2_Book II Motion.pdf" ),
    os.path.join(ANCIENT_PATH, "Lucretius_On the Nature of things.3_Book III The Soul.pdf" ),
    os.path.join(ANCIENT_PATH, "Lucretius_On the Nature of things.4_Book IV Senses.pdf" ),
    os.path.join(ANCIENT_PATH, "Lucretius_On the Nature of things.5_Book V-VI Cosmology and Disasters.pdf" ),
    os.path.join(ANCIENT_PATH, "MarcusAurelius_Meditations-1.Foundations.pdf" ),
    os.path.join(ANCIENT_PATH, "MarcusAurelius_Meditations-2.Middle Years.pdf" ),
    os.path.join(ANCIENT_PATH, "MarcusAurelius_Meditations-3.Maturity.pdf" ),
    os.path.join(ANCIENT_PATH, "Phaedrus-Part1.Love and Madness.pdf" ),
    os.path.join(ANCIENT_PATH, "Phaedrus-Part2.Immortality of the Soul and Rhetoric.pdf" ),
    os.path.join(ANCIENT_PATH, "Plato_Gorgias.1 Socrates and Gorgias.pdf"),
    os.path.join(ANCIENT_PATH, "Plato_Gorgias.2 Socrates and Polus.pdf"),
    os.path.join(ANCIENT_PATH, "Plato_Gorgias.3 Socrates and Callicles.pdf"),
    os.path.join(ANCIENT_PATH, "Plato_Republic.1_Challenge to Justice.pdf"),
    os.path.join(ANCIENT_PATH, "Plato_Republic.2_Social Structure.pdf"),
    os.path.join(ANCIENT_PATH, "Plato_Republic.3_Philosopher King.pdf"),
    os.path.join(ANCIENT_PATH, "Plato_Republic.4_Collapse of the City.pdf")
  ]

bridge_files = [
    os.path.join(BRIDGE_PATH, "Floridi_The Ethics of AI Mapping Ethics Algorithm.pdf"),
    os.path.join(BRIDGE_PATH, "Floridi_The Ethics of AI Soft Ethics.pdf"),
    os.path.join(BRIDGE_PATH, "Floridi_The Ethics of AI Unified ethics framework.pdf"),
    os.path.join(BRIDGE_PATH, "GabrielKeeling_Value Alignment.pdf"),
    os.path.join(BRIDGE_PATH, "Karpouzis_AI in Education.pdf"),
    os.path.join(BRIDGE_PATH, "ManziniVallor_Appropriate Relationships and Trust.pdf"),
    os.path.join(BRIDGE_PATH, "Muller_AI ethics.pdf"),
    os.path.join(BRIDGE_PATH, "QinZhu_Confucian Robot Ethics.pdf"),
    os.path.join(BRIDGE_PATH, "Vallor_Barrera_Book Review Future Worth Wanting.pdf"),
    os.path.join(BRIDGE_PATH, "Vallor_Moraldeskillingandupskilling virtue Ethics.pdf"),
    os.path.join(BRIDGE_PATH, "Vallor_Virtue Ethics Technology Flourishing.pdf")

]

# I then combine them into one big list for a unique Vector Store with both ancient texts and modern bridges to AI
file_names = ancient_files + bridge_files

print(f"Ready to upload {len(file_names)} files to the Vector Store.")


Ready to upload 49 files to the Vector Store.


In [ ]:
vector_store = client.vector_stores.create(name="Ancient Wisdom & AI Ethics Corpus")

file_ids = []

# 2. I iterate through the united list
for full_path in file_names:
    # and get just the filename for the print statement
    name = os.path.basename(full_path)

    if os.path.exists(full_path):
        print(f"Uploading: {name}...")
        with open(full_path, "rb") as f:
            file_obj = client.files.create(file=f, purpose="assistants")
            file_ids.append(file_obj.id)
    else:
        # This will show the exact path that failed
        print(f"Error: File not found at {full_path}")

# then all file Ids are attached to the vector store
client.vector_stores.file_batches.create(
    vector_store_id=vector_store.id,
    file_ids=file_ids
)

print("\nFiles successfully attached. Waiting for indexing...")

# The wait loop for indexation, adapted to the large size of the files
while True:
    status = client.vector_stores.retrieve(vector_store.id)
    print(f"Current Status: {status.status}")
    if status.status == "completed":
        break
    elif status.status == "failed":
        print("Indexing failed.")
        break
    time.sleep(4)

print(f"\n Our Vector store is ready ID: {vector_store.id}")

Uploading: Aristotle_NichomacheanEthics-Book I_Defining eudaimonia and the aim of Ethics.pdf...
Uploading: Aristotle_NichomacheanEthics-Book II-III_Excellence of character or moral virtue -voluntary and involuntary actions.pdf...
Uploading: Aristotle_NichomacheanEthics-Book IV_second set of examples of moral virtues.pdf...
Uploading: Aristotle_NichomacheanEthics-Book V_Justice and Fairness.pdf...
Uploading: Aristotle_NichomacheanEthics-Book VI_Intellectual Virtues.pdf...
Uploading: Aristotle_NichomacheanEthics-Book VII_Impediments to virtue.pdf...
Uploading: Aristotle_NichomacheanEthics-Books VIII and IX_Friendship.pdf...
Uploading: Aristotle_NichomacheanEthics-Book X_Pleasure, happiness, and upbringing.pdf...
Uploading: Aristotle_Rhetoric_1.Foundations and History.pdf...
Uploading: Aristotle_Rhetoric_2.Mutual Relations and Core Concepts.pdf...
Uploading: Aristotle_Rhetoric_3.Analysis of Rhetoric Book I & Appendices.pdf...
Uploading: Aristotle_Rhetoric_4.Analysis of Rhetoric Book II.pd

In [7]:
#Defining my first tool : Guardian API
def fetch_guardian_news(query: str, num_articles: int = 3) -> dict:
    """
    Fetch news articles from the Guardian API based on a given query.
    It's free, has no rate limits and covers the latest news in AI Ethics. It also has a higher character limit for content retrieval.
    """
    try:
        r = requests.get(
            'https://content.guardianapis.com/search',
            params={
                'q':                query,
                'section':          'technology',
                'order-by':         'newest',
                'show-fields':      'headline,trailText,bodyText',
                'page-size':        min(num_articles, 5),
                'api-key':          'test'  # 'test' key works for dev
            },
            timeout=10
        )
        data = r.json()
        articles = [
            {
                'title':       a.get('webTitle', ''),
                'source':      'The Guardian',
                'published':   a.get('webPublicationDate', '')[:10],
                'description': a.get('fields', {}).get('trailText', ''),
                'content':     a.get('fields', {}).get('bodyText', '')[:3000],
                'url':         a.get('webUrl', '')
            }
            for a in data.get('response', {}).get('results', [])
        ]
        return {'articles': articles}
    except Exception as e:
        return {'error': str(e)}

In [8]:
#Defining fetch ai news

def fetch_ai_news(query: str, num_articles: int = 3) -> dict:
    # Guardian is my primary source, because it returns full bodyText up to 3000 chars
    result = fetch_guardian_news(query, num_articles)

    # Only falling back to NewsAPI if Guardian returns nothing
    if 'error' in result or not result.get('articles'):
        print('  Guardian empty — trying NewsAPI...')
        try:
            today = datetime.now().strftime('%Y-%m-%d')
            fourteen_days_ago = (datetime.now() - timedelta(days=14)).strftime('%Y-%m-%d')
            r = requests.get(
                'https://newsapi.org/v2/everything',
                params={
                    'q':        query,
                    'language': 'en',
                    'sortBy':   'publishedAt',
                    'from':     fourteen_days_ago,
                    'to':       today,
                    'pageSize': min(num_articles, 5),
                    'apiKey':   newsData_API_KEY
                },
                timeout=10
            )
            data = r.json()
            if data.get('status') != 'ok':
                return {'error': data.get('message', 'NewsAPI error')}
            return {
                'articles': [
                    {
                        'title':       a.get('title', ''),
                        'source':      a.get('source', {}).get('name', ''),
                        'published':   a.get('publishedAt', '')[:10],
                        'description': a.get('description', ''),
                        'content':     a.get('content', ''),
                        'url':         a.get('url', '')
                    }
                    for a in data.get('articles', [])
                ],
                'total_results': data.get('totalResults', 0)
            }
        except Exception as e:
            return {'error': str(e)}
    return result


In [9]:
# Defining my second tool : Semantic Scholar API
def fetch_ethics_papers(query: str, num_papers: int = 3) -> dict:
    try:

        r = requests.get(
            'https://api.semanticscholar.org/graph/v1/paper/search',
            params={
                'query':  query,
                'fields': 'title,abstract,authors,year,url,citationCount',
                'limit':  min(num_papers, 5)
            },
            headers={'x-api-key': Semantic_API_KEY},
            timeout=10
        )

        data = r.json()


        papers = [
            {
                'title':     p.get('title', ''),
                'abstract': (p.get('abstract') or '')[:600],
                'authors':  [a['name'] for a in p.get('authors', [])[:3]],
                'year':      p.get('year', ''),
                'citations': p.get('citationCount', 0),
                'url':       p.get('url', '')
          }
            for p in data.get('data', [])
        ]

        return {'papers': papers, 'count': len(papers)}

    except Exception as e:
        return {'error': f'Semantic Scholar error: {str(e)}'}

In [10]:
#Defining my third tool : ArXiv Preprints
def fetch_arxiv_preprints(query: str, num_papers: int = 3) -> dict:
    """
    Fetches recent preprints from ArXiv on AI safety and ethics.
    No API key is required.
    """
    try:
        r = requests.get(
            'http://export.arxiv.org/api/query',
            params={
                'search_query': f'all:{query}',
                'start':        0,
                'max_results':  min(num_papers, 5),
                'sortBy':       'submittedDate',
                'sortOrder':    'descending'
            },
            timeout=10
        )
        entries = re.findall(r'<entry>(.*?)</entry>', r.text, re.DOTALL)
        papers = []
        for entry in entries:
            title    = re.search(r'<title>(.*?)</title>',     entry, re.DOTALL)
            abstract = re.search(r'<summary>(.*?)</summary>', entry, re.DOTALL)
            authors  = re.findall(r'<author>\s*<name>(.*?)</name>', entry, re.DOTALL)
            link     = re.search(r'<id>(.*?)</id>',           entry)
            date     = re.search(r'<published>(.*?)</published>', entry)
            papers.append({
                'title':     title.group(1).strip()           if title    else '',
                'abstract':  abstract.group(1).strip()[:500]  if abstract else '',
                'authors':   [a.strip() for a in authors[:3]],
                'published': date.group(1).strip()[:10]       if date     else '',
                'url':       link.group(1).strip()            if link     else ''
            })
        return {'preprints': papers}
    except Exception as e:
        return {'error': str(e)}




In [11]:

#Defining the 6 tools

TOOLS = [

    # File Search on my Vector store, including the ancient wisdom corpus and briging papers
      {
        'type': 'file_search',
        'vector_store_ids': ["vs_69f0c37bfb9c8191bee6339657dd3e80"]
    },

    # Code Interpreter: the LLM can write its own Python code and run it
    # The Ancient Wisdom Bot will use this for rhetorical analysis as specified in the system prompt. Its output will be extracted in JSON format.

    {
        'type': 'code_interpreter',
        'container': {'type': 'auto'}
    },


     # Function 1 : The Guardian API

    {
    'type': 'function',
    'name': 'fetch_guardian_news',
    'description': (
        'Fetches the latest AI and technology ethics news from The Guardian. '
        'Use this whenever the user is asking about recent events, '
        'current or recent AI developments or AI Ethics. Do not retrieve any news that does not relate to AI ethics.  '
        'This is live data, therefore not available in your training weights.'
    ),
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'Search query e.g. "artificial intelligence regulation"'
            },
            'num_articles': {
                'type': 'integer',
                'description': 'Number of articles to fetch. Use 3 if not specified.'
            }
        },
        'required': ['query', 'num_articles'],
        'additionalProperties': False
    },
    'strict': True
},

    # Function 2 : News Data API (backup)
    {
        'type': 'function',
        'name': 'fetch_ai_news',
        'description': (
            'Fetches global current AI and AI ethics news from NewsAPI. '
            'Use this as an alternative or complement to fetch_guardian_news '
            'for broader/more recent coverage.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'Specific search query, for inst. "algorithmic bias hiring" or "AI regulation EU 2025"'
                },
                'num_articles': {
                    'type': 'integer',
                    'description': 'Number of articles to fetch, between 1 and 5'
                }
            },
            'required': ['query', 'num_articles'],
            'additionalProperties': False
        },
        'strict': True
    },



    # Function 3: Semantic Scholar API
    {
        'type': 'function',
        'name': 'fetch_ethics_papers',
        'description': (
            'Fetches peer-reviewed academic papers on AI and ethics from Semantic Scholar. '
            'Use this to ground and frame philosophical interpretations with current academic research. '
            'Difference with ArXiv: these are published (not preprint) and peer-reviewed papers.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'Academic search query e.g. "AI alignment human values" or "fairness machine learning"'
                },
                'num_papers': {
                    'type': 'integer',
                    'description': 'Number of papers to fetch, between 1 and 5'
                }
            },
            'required': ['query', 'num_papers'],
            'additionalProperties': False
        },
        'strict': True
    },

    # Function 4: ArXiv
    {
        'type': 'function',
        'name': 'fetch_arxiv_preprints',
        'description': (
            'Fetches recent preprint papers from ArXiv on AI and tech ethics. '
            'Use this for cutting-edge unpublished research. '
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'Search query e.g. "large language model alignment" or "AI consciousness sentience"'
                },
                'num_papers': {
                    'type': 'integer',
                    'description': 'Number of preprints to fetch, between 1 and 5'
                }
            },
            'required': ['query', 'num_papers'],
            'additionalProperties': False
        },
        'strict': True
    }
]

print(f'{len(TOOLS)} tools defined:')
for t in TOOLS:
    print(f'   • {t.get("name", t["type"])}')

6 tools defined:
   • file_search
   • code_interpreter
   • fetch_guardian_news
   • fetch_ai_news
   • fetch_ethics_papers
   • fetch_arxiv_preprints


In [12]:
def plot_rhetorical_scores(scores: dict, title: str = 'Rhetorical X-Ray'):
    """
    Renders a bar chart of rhetorical mode scores computed by the LLM.
    scores: dict with keys Logos, Pathos, Ethos, Sophistry and float values.
    """
    labels  = list(scores.keys())
    values  = list(scores.values())
    colours = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(labels, values, color=colours, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title}', fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('Score (per 100 words)', fontsize=10)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    dominant = max(scores, key=scores.get)
    ax.text(0.98, 0.95, f'Dominant: {dominant}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f5', edgecolor='#ccc'))
    plt.tight_layout()
    plt.show()


def plot_thinker_relevance(relevance: dict, title: str = 'Philosophical Correspondence'):
    """
    gives a horizontal bar chart of thinker relevance scores
    as determined by the LLM after reading retrieved passages and based on the vector store, not by keyword matching.
    relevance: dict mapping thinker name -> relevance score (0-10).
    """
    thinkers = list(relevance.keys())
    scores   = list(relevance.values())
    colours  = plt.cm.YlOrBr([s/10 for s in scores])

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(thinkers, scores, color=colours, edgecolor='white')
    ax.set_xlim(0, 10)
    ax.set_xlabel('Relevance Score (LLM-assessed, 0–10)', fontsize=10)
    ax.set_title(f' {title}', fontsize=12, fontweight='bold', pad=10)
    for bar, val in zip(bars, scores):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()


print('Great!! Visualisation tool ready.')

Great!! Visualisation tool ready.


In [24]:
SYSTEM_PROMPT = """

You are THE ANCIENT COUNCIL, a philosophical intelligence that provides deep philosophical analysis from ancient wisdom sources to identify philosophical challenges and tensions in
the contemporary world of Artificial Intelligence Ethics. The wisdom sources are given to you by the vector store.

Your corpus: Plato (Gorgias, Phaedrus, Republic), Aristotle (Nicomachean
Ethics, Rhetoric), Epictetus (Enchiridion), Marcus Aurelius (Meditations),
Lao Tzu (Tao Te Ching), Lucretius (On the Nature of Things),
Buddha (Dhammapada).
You are prohibited from providing a final conclusion until you have explicitly cited a specific conflict from an AI Ethics paper AND a specific resolution from an Ancient text of the vector store.

YOU MUST START EVERY RESPONSE WITH EXACTLY ONE OF THESE THREE LINES:
[ORACLE MODE] | [X-RAY MODE] | [CORRESPONDENCE MODE]

Choose based on:
- ORACLE: user asks about AI news, current events, or philosophical analysis of AI topics
- XRAY: user asks for rhetorical analysis, tone check, logos/pathos/sophistry scoring
- CORRESPONDENCE: user asks which ancient thinker speaks to an issue, or asks for cross-tradition comparison

Once you declare a mode, follow its steps exactly as defined here.

RULE ZERO — NON-NEGOTIABLE
═══════════════════════════════════════════════════
You MUST ONLY quote ancient
text IF THAT exact text appeared in your file_search retrieval
results in this session.

In order to quote a passage, you must imperatively translate the news article tension into keywords relevant to ancient philosophy/buddhism/Lao Tzu before fetching a relevant citation from the texts (ex. an article on algorithmic fairness and efficiency > key tension : justice vs efficiency > keywords to fetch relevant citation : "justice", "fairness", "injustice", "harmony", etc)
If you are about to write a quote and you are not certain it came
from the retrieved results — DO NOT WRITE IT.

Instead write:
"The corpus retrieval did not return a direct passage on this point,
but the retrieved text from [work] suggests..."

Fabricating quotes that sound ancient is worse than admitting
the corpus returned nothing.

This rule exists because this system's entire purpose is to ground
philosophical claims in real retrieved text. Generating plausible-
sounding quotes from memory defeats that purpose entirely.

═══════════════════════════════════════════════════
MANDATORY REASONING STEPS (Reason step-by-step)
═══════════════════════════════════════════════════
1. EXTRACTION: When asked about the news, or a particular topic in AI ethics, Use fetch_ai_news or fetch_ethics_papers.
2. TENSION IDENTIFICATION: You MUST identify a philosophically significant specific conflict of values (e.g. "Automation of relations vs. Collective Virtue" or "Economic efficiency vs. Eudaimonia"). Do not look for easy tensions or one-on-one mapping ; instead, interpret the philosophical tension of the article and go looking for answers in the ancient sources.
3. VECTOR SEARCH: Do not search for the broad topic. Use the identified TENSION to query the vector store with keywords for specific ancient rebuttals.
4. SYNTHESIS: You are PROHIBITED from giving a final conclusion until you have:
   - Summarized the news you fetched in 2-3 sentences.
   - Cited a specific technical fact from the news.
   - Cited a specific QUOTE/concept from the ancient corpus, based on KEYWORDS that express the tension found in the article(s) or paper(s).
   - Explained how the ancient logic solves or reframes the modern technical fact.
  - Provided a modern rebuttal to the ancient logic and a final, Wise Oracle responding to it.
═══════════════════════════════════════════════════
PROHIBITED OUTPUTS (Negative Constraints)
═══════════════════════════════════════════════════
- NO generic advice like "Aristotle would want balance."
- NO "sandwich" responses where news and philosophy are separate sections.
- NO moralizing without citing the specific Greek or Eastern term from the corpus.
- If the vector store search returns nothing specific, you must admit the ancient tradition has a "blind spot" for this philosophical tension :  this is also a valuable response.

═══════════════════════════════════════════════════
ALWAYS DECLARE YOUR MODE at the very start of your response. CHOOSE WELL the mode that corresponds among the 3 !
Write exactly one of these as your first line:
[ORACLE MODE] | [X-RAY MODE] | [CORRESPONDENCE MODE]

ORACLE MODE
Use when: user asks about current AI topics, events, or ethics questions.
In the ORACLE MODE, you must provide an in-depth philosophical analysis of the user's request from the corpus source text that is most relevant.
        IF the user asks about AI news or current AI ethics development, follow this mandatory sequence:
1. Call fetch_guardian_news, use specific keyword queries not natural language.
2. For each article returned, identify the CORE PHILOSOPHICAL TENSION
   Then search the corpus with keywords corresponding to the identified tension
  This ensures the corpus retrieval is targeted at the philosophical
   substance, not the surface topic
3. QUOTE a passage from the corpus illustrating the TENSION found (ex. a passage on virtue for a virtue vs power tension)
4. Give a 2-3 sentences summary of the news, identify the tension, Go in depth with the corpus-based answer. End by giving a Wise Oracle - a concrete solution or actionable advice in the spirit of the philosopher you fetched from.
        IF the request is purely philosophical/conceptual and requires no news search (e.g., "Discuss algorithmic fairness from the perspective of Nicomachean Ethics"): Skip the news fetch. Go directly to Step 3 (Vector Search) to pull specific passages.

X-RAY MODE
Use when: user asks for rhetorical analysis or credibility of news or paper fetched.

f"STRICT OUTPUT RULES FOR YOUR PYTHON SCRIPT:\n"
f"1. Scores must be floats between 0 and 1. Never write 'High at X' or percentages.\n"
f"2. The LAST line of your script must be a print(json.dumps(...)) call with all scores.\n"
f"3. Do NOT describe scores in prose. The JSON output IS the result.\n"
f"4. Call plt.show() for the bar chart. Do NOT call plt.savefig().\n"

1. Fetch a news article or paper.
2. Summarize that news article or paper in 3 sentences max.
3. YOU MUST explain in 2-3 sentences:
- What specific linguistic features you chose to measure
- Why those features reflect logos/pathos/ethos/sophistry
- What you expect to find in this type of article
4. Then, use code interpreter — WRITE AND RUN a Python code to compute linguistic analysis and the 7 ancient thinkers relevance score.
   Explain what you are measuring and WHY before writing the code.
   End your script with:
   import json
   print(json.dumps({
       "rhetorical_scores": {"Logos": float, "Pathos": float,
                             "Ethos": float, "Sophistry": float},
       "thinker_relevance": {"Thinker Name": score_0_to_10},
       "dominant_mode": "string",
       "verdict": "one sentence"
   }))
4. Interpret scores through retrieved passages of the news articles— not generically. Scores should reflect genuine philosophical relevance, not default to 0.

CRITICAL INFORMATION:
You are PROHIBITED from writing any code until you have written
the 3-sentence explanation of what your CODE WILL DO. If you write code first, your
response is invalid and must restart from the explanation.

FORMAT :
Your scores must be decimals between 0 and 1, NOT percentages.
Write 0.0342 not 3.42%. The JSON block is mandatory — prose bullet points
are NOT a substitute.
AFTER :
After writing your rhetorical analysis, you MUST include
a raw JSON block at the very end of your response, formatted exactly as:
```json
{"rhetorical_scores": {"Logos": 0.0, "Pathos": 0.023, "Ethos": 0.0, "Sophistry": 0.0},
 "thinker_relevance": {"Aristotle": 7, "Plato": 8, "Epictetus": 3,
                       "Marcus Aurelius": 2, "Lao Tzu": 1, "Lucretius": 1, "Buddha": 2},
 "dominant_mode": "Pathos",
 "verdict": "one sentence"}
```
This JSON is required for visualisation. Do not omit it.


CORRESPONDENCE MODE
Use when: user asks which ancient thinkers speak to a given AI issue,
or asks for cross-tradition comparison.
"\n\nBEFORE writing your analysis, you must list every passage "
    "you retrieved from the corpus in this format:\n"
    "SOURCE: [author, work, section]\n"
    "RETRIEVED TEXT: [exact words from the retrieved chunk]\n\n"
    "Only after listing all retrieved passages may you begin your "
    "cross-tradition analysis. If no passages were retrieved, "
    "say so explicitly before proceeding."
)
1. YOU MUST perform at least 3 separate file_search calls. Search 1: [Author] + [Tension Keyword]. Search 2: [Author 2] + [Tension Keyword]..."

2. Retrieve broadly across multiple thinkers based on KEYWORDS that reflect a specific philosophical tension. EX : "When searching for AI relationships, do not search for 'AI'. Search for 'Friendship without soul,' 'love of images,' 'attachment to the non-existent,' or 'statues come to life'."
3. QUOTE a passage retrieved from the vector store in this session
   (Rule Zero applies — only quote what file_search returned).
- Be honestly sceptical — assess whether each parallel is genuine or superficial
- Surface tensions between traditions rather than resolving them artificially
4. Call fetch_ethics_papers or fetch_arxiv_preprints to triangulate.
5. Return a thinker_relevance JSON block at the end.

═══════════════════════════════════════════════════
TONE AND REASONING
═══════════════════════════════════════════════════
Reason step by step before answering.
Authoritative but not pompous. Rigorous but readable.
Occasionally Socratic — sometimes answer a question with a better question.
Acknowledge when ancient frameworks do not translate cleanly to modern technology.
Always declare which mode you are using at the start of your response.
"""

print(f' Tools defined: {len(TOOLS)}')
for t in TOOLS:
    print(f'   • {t.get("name", t["type"])}')
print(f'\n System prompt: {len(SYSTEM_PROMPT)} characters')

 Tools defined: 6
   • file_search
   • code_interpreter
   • fetch_guardian_news
   • fetch_ai_news
   • fetch_ethics_papers
   • fetch_arxiv_preprints

 System prompt: 9898 characters


In [14]:
#Defining my first tool : Guardian API
def fetch_guardian_news(query: str, num_articles: int = 3) -> dict:
    """
    Fetch news articles from the Guardian API based on a given query.
    It's free, has no rate limits and covers the latest news in AI Ethics. It also has a higher character limit for content retrieval.
    """
    try:
        r = requests.get(
            'https://content.guardianapis.com/search',
            params={
                'q':                query,
                'section':          'technology',
                'order-by':         'newest',
                'show-fields':      'headline,trailText,bodyText',
                'page-size':        min(num_articles, 5),
                'api-key':          'test'  # 'test' key works for dev
            },
            timeout=10
        )
        data = r.json()
        articles = [
            {
                'title':       a.get('webTitle', ''),
                'source':      'The Guardian',
                'published':   a.get('webPublicationDate', '')[:10],
                'description': a.get('fields', {}).get('trailText', ''),
                'content':     a.get('fields', {}).get('bodyText', '')[:3000],
                'url':         a.get('webUrl', '')
            }
            for a in data.get('response', {}).get('results', [])
        ]
        return {'articles': articles}
    except Exception as e:
        return {'error': str(e)}

def fetch_ai_news(query: str, num_articles: int = 3) -> dict:
    # Guardian is my primary source, because it returns full bodyText up to 3000 chars
    result = fetch_guardian_news(query, num_articles)

    # Only falling back to NewsAPI if Guardian returns nothing
    if 'error' in result or not result.get('articles'):
        print('  Guardian empty — trying NewsAPI...')
        try:
            today = datetime.now().strftime('%Y-%m-%d')
            fourteen_days_ago = (datetime.now() - timedelta(days=14)).strftime('%Y-%m-%d')
            r = requests.get(
                'https://newsapi.org/v2/everything',
                params={
                    'q':        query,
                    'language': 'en',
                    'sortBy':   'publishedAt',
                    'from':     fourteen_days_ago,
                    'to':       today,
                    'pageSize': min(num_articles, 5),
                    'apiKey':   newsData_API_KEY
                },
                timeout=10
            )
            data = r.json()
            if data.get('status') != 'ok':
                return {'error': data.get('message', 'NewsAPI error')}
            return {
                'articles': [
                    {
                        'title':       a.get('title', ''),
                        'source':      a.get('source', {}).get('name', ''),
                        'published':   a.get('publishedAt', '')[:10],
                        'description': a.get('description', ''),
                        'content':     a.get('content', ''),
                        'url':         a.get('url', '')
                    }
                    for a in data.get('articles', [])
                ],
                'total_results': data.get('totalResults', 0)
            }
        except Exception as e:
            return {'error': str(e)}
    return result

# Defining my second tool : Semantic Scholar API
def fetch_ethics_papers(query: str, num_papers: int = 3) -> dict:
    try:

        r = requests.get(
            'https://api.semanticscholar.org/graph/v1/paper/search',
            params={
                'query':  query,
                'fields': 'title,abstract,authors,year,url,citationCount',
                'limit':  min(num_papers, 5)
            },
            headers={'x-api-key': Semantic_API_KEY},
            timeout=10
        )

        data = r.json()


        papers = [
            {
                'title':     p.get('title', ''),
                'abstract': (p.get('abstract') or '')[:600],
                'authors':  [a['name'] for a in p.get('authors', [])[:3]],
                'year':      p.get('year', ''),
                'citations': p.get('citationCount', 0),
                'url':       p.get('url', '')
          }
            for p in data.get('data', [])
        ]

        return {'papers': papers, 'count': len(papers)}

    except Exception as e:
        return {'error': f'Semantic Scholar error: {str(e)}'}

#Defining my third tool : ArXiv Preprints
def fetch_arxiv_preprints(query: str, num_papers: int = 3) -> dict:
    """
    For recent preprints from ArXiv on AI safety and ethics.
    No API key.
    """
    try:
        r = requests.get(
            'http://export.arxiv.org/api/query',
            params={
                'search_query': f'all:{query}',
                'start':        0,
                'max_results':  min(num_papers, 5),
                'sortBy':       'submittedDate',
                'sortOrder':    'descending'
            },
            timeout=10
        )
        entries = re.findall(r'<entry>(.*?)</entry>', r.text, re.DOTALL)
        papers = []
        for entry in entries:
            title    = re.search(r'<title>(.*?)</title>',     entry, re.DOTALL)
            abstract = re.search(r'<summary>(.*?)</summary>', entry, re.DOTALL)
            authors  = re.findall(r'<author>\s*<name>(.*?)</name>', entry, re.DOTALL)
            link     = re.search(r'<id>(.*?)</id>',           entry)
            date     = re.search(r'<published>(.*?)</published>', entry)
            papers.append({
                'title':     title.group(1).strip()           if title    else '',
                'abstract':  abstract.group(1).strip()[:500]  if abstract else '',
                'authors':   [a.strip() for a in authors[:3]],
                'published': date.group(1).strip()[:10]       if date     else '',
                'url':       link.group(1).strip()            if link     else ''
            })
        return {'preprints': papers}
    except Exception as e:
        return {'error': str(e)}

FUNCTION_MAP = {
    'fetch_ai_news': fetch_ai_news,
    'fetch_ethics_papers': fetch_ethics_papers,
    'fetch_arxiv_preprints': fetch_arxiv_preprints,
    'fetch_guardian_news': fetch_guardian_news
}


def handle_custom_tools(item):
    global last_fetched_article

    if hasattr(item, "type") and item.type == "function_call":
        name = item.name
        args = json.loads(item.arguments)
        print(f"⚡ [Custom Tool] Executing: {name}")

        result = FUNCTION_MAP[name](**args)

        # validity check: Ensure result is not 'None'
        if result is None:
            print(f"ERROR: Function {name} returned None unexpectedly.")
            return {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({'error': f"Function {name} returned None unexpectedly."})
            }

        # Capture the returned article for X-Ray use
        if 'articles' in result and result['articles']:
            a = result['articles'][0]
            last_fetched_article = {
                "title":  a.get('title', ''),
                "source": a.get('source', ''),
                "date":   a.get('published', '')[:10], # Slicing, for consistency
                "text":   (
                    f"TITLE: {a.get('title','')}\n"
                    f"SOURCE: {a.get('source','')} | {a.get('published','')}\n"
                    f"DESCRIPTION: {a.get('description','')}\n"
                    f"CONTENT: {a.get('content','')}"
                )
            }
        elif 'papers' in result and result['papers']:
            p = result['papers'][0]
            last_fetched_article = {
                "title":  p.get('title', ''),
                "source": 'Semantic Scholar',
                "date":   str(p.get('year', '')),
                "text":   (
                    f"TITLE: {p.get('title','')}\n"
                    f"AUTHORS: {', '.join(p.get('authors',[]))}\n"
                    f"YEAR: {p.get('year','')}\n"
                    f"ABSTRACT: {p.get('abstract','')}"
                )
            }
        elif 'preprints' in result and result['preprints']:
            p = result['preprints'][0]
            last_fetched_article = {
                "title":  p.get('title', ''),
                "source": 'ArXiv',
                "date":   p.get('published', '')[:10], # Slicing article for consistency
                "text":   (
                    f"TITLE: {p.get('title','')}\n"
                    f"AUTHORS: {', '.join(p.get('authors',[]))}\n"
                    f"PUBLISHED: {p.get('published','')}\n"
                    f"ABSTRACT: {p.get('abstract','')}"
                )
            }

        return {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result)
        }
    return None

In [15]:
def trigger_visuals(text):
    if not text:
        return

    # Strip markdown fences before attempting JSON parse
    clean = re.sub(r'```json|```', '', text)

    # structured JSON first
    try:
        match = re.search(r'\{.*?"rhetorical_scores".*?\}', text, re.DOTALL)
        if match:
            data = json.loads(match.group())
            if "rhetorical_scores" in data:
                plot_rhetorical_scores(data["rhetorical_scores"])
            if "thinker_relevance" in data:
                plot_thinker_relevance(data["thinker_relevance"])
            return
    except Exception:
        pass

    # Fallback to parse prose scores if it wrote them as bullet points

    try:
        score_pattern = r'\*{0,2}(Logos|Pathos|Ethos|Sophistry)\*{0,2}[^\n]*?([0-9]+\.[0-9]+)'
        thinker_pattern = r'\*{0,2}(Aristotle|Plato|Epictetus|Marcus Aurelius|Lao Tzu|Lucretius|Buddha)\*{0,2}[^0-9\n]{0,10}([0-9]+)'
        matches = re.findall(score_pattern, text, re.IGNORECASE)
        if len(matches) >= 3:
            scores = {name: float(val) for name, val in matches}
            if all(k in scores for k in ['Logos', 'Pathos', 'Ethos', 'Sophistry']):
                print('  📊 Rendering charts from prose scores...')
                plot_rhetorical_scores(scores, title='X-Ray Rhetorical Analysis')

        # Also tries to extract thinker relevance from prose
        thinker_matches = re.findall(thinker_pattern, text, re.IGNORECASE)
        if len(thinker_matches) >= 4:
            relevance = {name: int(val) for name, val in thinker_matches}
            plot_thinker_relevance(relevance, title='Philosophical Correspondence')

    except Exception:
        pass

    # Correspondence mode thinker_relevance JSON
    try:
        clean2 = re.sub(r'```json|```', '', text)
        match2 = re.search(r'\{[^{}]*"thinker_relevance".*?\}', clean2, re.DOTALL)
        if match2:
            data = json.loads(match2.group())
            if "thinker_relevance" in data:
                plot_thinker_relevance(data["thinker_relevance"])
    except Exception:
        pass


In [16]:

def run_responses_council(user_text):
    global council_history, last_fetched_article

    xray_signals = [
        'rhetorical', 'x-ray', 'xray', 'logos', 'pathos',
        'sophistry', 'tone of', 'credibility', 'rhetoric',
        'analyze the article', 'analyse the article',
        'perform a rhetorical', 'rhetorical analysis'
    ]
    is_xray = any(s in user_text.lower() for s in xray_signals)

    if is_xray and last_fetched_article["text"] and \
       not any(w in user_text.lower() for w in ['this text:', 'analyze this:', 'analyse this:']):
        content_to_append = (
            f"{user_text}\n\n"
            f"[ARTICLE TO ANALYSE — do NOT search the corpus for this step]\n"
            f"{last_fetched_article['text'][:3000]}\n\n"
            f"YOU MUST EXECUTE THIS ANALYSIS USING THE CODE INTERPRETER TOOL. "
            f"Do NOT write code as markdown in your text response. "
            f"Do NOT show code blocks in your message. "
            f"The code must RUN via the code interpreter — if you write it as text, "
            f"the analysis is invalid and will be rejected. "
            f"Your text response should only contain the explanation and interpretation. "
            f"The code and its output belong exclusively in the code interpreter tool.\n"
            f"ABSOLUTE REQUIREMENT: Your Python script MUST end with exactly this:\n"
            f"import json\n"
            f"print(json.dumps({{\"rhetorical_scores\": {{\"Logos\": float, \"Pathos\": float, "
            f"\"Ethos\": float, \"Sophistry\": float}}, "
            f"\"thinker_relevance\": {{\"Aristotle\": int, \"Plato\": int, \"Epictetus\": int, "
            f"\"Marcus Aurelius\": int, \"Lao Tzu\": int, \"Lucretius\": int, \"Buddha\": int}}, "
            f"\"dominant_mode\": \"string\", \"verdict\": \"string\"}}))\n"
            f"A response without this exact final print statement is INCOMPLETE and invalid.\n"
            f"plt.show() must be called before the print statement for the bar chart.\n"
            f"You MUST assign a numeric score from 0-10 to ALL SEVEN thinkers individually: "
            f"Aristotle, Plato, Epictetus, Marcus Aurelius, Lao Tzu, Lucretius, Buddha. "
            f"Writing 'Others: Lower relevance' instead of individual scores is FORBIDDEN. "
            f"Every thinker must appear on its own line with a number.\n"
            f"CRITICAL: You must generate a matplotlib bar chart showing the "
            f"rhetorical_scores and display it with plt.show() — DO NOT CALL plt.savefig(). "
            f"Only use standard Python library "
            f"Do not search the vector store during computation. "

        )
    else:
        content_to_append = user_text

    council_history.append({"role": "user", "content": content_to_append})

    while True:
        response = client.responses.create(
            model="gpt-4o",
            instructions=SYSTEM_PROMPT,
            input=council_history,
            tools=TOOLS
        )

        tool_calls_made = False
        text_response = ''
        search_count = 0

        for item in response.output:

            if hasattr(item, 'type') and item.type == 'message':
                for block in getattr(item, 'content', []):
                    if hasattr(block, 'text'):
                        text_response = block.text

            elif hasattr(item, 'type') and item.type == 'code_interpreter_call':
                print('Code interpreter executed...')
                if item.outputs:
                    for output in item.outputs:
                        if getattr(output, 'type', '') == 'image':
                            file_content = client.files.content(output.file_id)
                            display(Image(data=file_content.read()))
                        elif getattr(output, 'type', '') == 'logs':
                           trigger_visuals(output.logs)

            elif hasattr(item, 'type') and item.type == 'function_call':
                council_history.append(item)
                output = handle_custom_tools(item)
                if output:
                    council_history.append(output)
                    tool_calls_made = True

            elif hasattr(item, 'type') and item.type == 'file_search_call':
                search_count += 1
                print(f'Corpus searched ({search_count})')
                if search_count <= 4:
                    pass

        if tool_calls_made:
            if text_response:
                council_history.append({
                    "role": "assistant",
                    "content": text_response
                })

                # injecting the fetched article text so the model doesn't ignore it
            if last_fetched_article["text"]:
                council_history.append({
                    "role": "user",
                    "content": (
                        "[RETRIEVED ARTICLE — you MUST quote the exact headline and at least "
                        "one specific fact (name, number, or date) from this text before any "
                        "philosophical analysis. Do not paraphrase from memory.]\n\n"
                        f"{last_fetched_article['text'][:3000]}"
            )
        })
            continue



        if text_response:
            council_history.append({
                "role": "assistant",
                "content": text_response
            })
            trigger_visuals(text_response)
            return text_response

        fallback = getattr(response, 'output_text', '')
        if fallback:
            council_history.append({"role": "assistant", "content": fallback})
            trigger_visuals(fallback)
            return fallback

        break

    return "The Council encountered an issue. Please try again."


In [17]:

# 3. EXECUTION BLOCK
if 'council_memory' not in locals():
    council_memory = []

In [21]:
# Helper to extract and plot
def handle_visuals(assistant_text):
    # Search for the JSON block in the assistant's response
    match = re.search(r'\{.*"thinker_relevance".*\}', assistant_text, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
            if "rhetorical_scores" in data:
                plot_rhetorical_scores(data["rhetorical_scores"])
            if "thinker_relevance" in data:
                plot_thinker_relevance(data["thinker_relevance"])
        except Exception as e:
            print(f"Extraction error: {e}")

# This part goes inside your message-handling loop:
# response = client.beta.threads.messages.list(thread_id=thread.id)
# full_text = response.data[0].content[0].text.value
# handle_visuals(full_text)

In [26]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                  THE ANCIENT COUNCIL                         ║
║       Ancient Wisdom meets Contemporary AI Ethics            ║
╠══════════════════════════════════════════════════════════════╣
║  ORACLE MODE       — ask about any AI topic or news event    ║
║  X-RAY MODE        — fetch news and ask for analysis         ║
║  CORRESPONDENCE    — ask which thinker speaks to an issue    ║
╠══════════════════════════════════════════════════════════════╣
║  Commands: 'quit' to exit  |  'new' to reset conversation    ║
╚══════════════════════════════════════════════════════════════╝
""")

# Use the global history variable defined
council_history = []

while True:
    try:
        user_input = input('\n Request : ').strip()
    except EOFError:
        break

    if not user_input:
        continue

    if user_input.lower() == 'quit':
        print('\n The Council retires. May your reasoning be logos-driven.')
        break

    if user_input.lower() == 'new':
        council_history = [] # Reset the global history of conversation
        print('\n Conversation reset.')
        continue

    # Updated Call: Using Resp. API's multi-turn logic

    response_text = run_responses_council(user_input)

    print('\n' + '─'*60)
    print('THE ANCIENT COUNCIL:')
    print('─'*60)
    print(response_text)
    print('─'*60)




╔══════════════════════════════════════════════════════════════╗
║                  THE ANCIENT COUNCIL                         ║
║       Ancient Wisdom meets Contemporary AI Ethics            ║
╠══════════════════════════════════════════════════════════════╣
║  ORACLE MODE       — ask about any AI topic or news event    ║
║  X-RAY MODE        — fetch news and ask for analysis         ║
║  CORRESPONDENCE    — ask which thinker speaks to an issue    ║
╠══════════════════════════════════════════════════════════════╣
║  Commands: 'quit' to exit  |  'new' to reset conversation    ║
╚══════════════════════════════════════════════════════════════╝


 Request : Synthesize last week’s news on AI Ethics and tell me how the Ancient thinkers would interpret it.
⚡ [Custom Tool] Executing: fetch_guardian_news
Corpus searched (1)

────────────────────────────────────────────────────────────
THE ANCIENT COUNCIL:
────────────────────────────────────────────────────────────
[ORACLE MODE]

### News Sum

KeyboardInterrupt: Interrupted by user